In [ ]:
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

data = pd.read_csv('/media/abanoub/Storage1/My-Github/UK-Train-Rides/Dataset/railway_after_analysis.csv')
data['Delayed?'] = data['Reason for Delay'].apply(lambda x: 1 if x != 'No Reason' else 0)

# Feature setting
features = [
    'Departure Station', 'Arrival Destination', 'Month', 'Journey Day', 'Departure Hour', 'day number'
]
X = data[features]
y = data['Delayed?']

ct = ColumnTransformer([
    ("ohe", OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), features)
])

X_transformed = ct.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, y, test_size=0.3, random_state=46
)

sm = SMOTE(random_state=46)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

model = RandomForestClassifier(n_estimators=120, max_depth=19, class_weight='balanced', random_state=46)
model.fit(X_train_res, y_train_res)

y_train_pred = model.predict(X_train_res)
print(f"Train accuracy: {accuracy_score(y_train_res, y_train_pred):.4f}")

y_test_pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_test_pred):.4f}")

print("\n=== Classification Report (Default Threshold) ===")
print(classification_report(y_test, y_test_pred, target_names=['No Delay', 'Delayed']))
print("\n=== Confusion Matrix (Default Threshold) ===")
print(confusion_matrix(y_test, y_test_pred))

y_test_probs = model.predict_proba(X_test)[:, 1]
threshold = 0.35
y_test_pred_tuned = (y_test_probs > threshold).astype(int)

print(f"\n=== Classification Report (Threshold={threshold}) ===")
print(classification_report(y_test, y_test_pred_tuned, target_names=['No Delay', 'Delayed']))
print(f"\n=== Confusion Matrix (Threshold={threshold}) ===")
print(confusion_matrix(y_test, y_test_pred_tuned))

model_package = {
    'model': model,
    'transformer': ct,
    'feature_names': features
}
with open('predict_delay.pkl', 'wb') as f:
    pickle.dump(model_package, f)

Train accuracy: 0.8875
Test accuracy: 0.9017

=== Classification Report (Default Threshold) ===
              precision    recall  f1-score   support

    No Delay       0.94      0.95      0.94      8258
     Delayed       0.63      0.59      0.61      1238

    accuracy                           0.90      9496
   macro avg       0.79      0.77      0.78      9496
weighted avg       0.90      0.90      0.90      9496


=== Confusion Matrix (Default Threshold) ===
[[7827  431]
 [ 502  736]]

=== Classification Report (Threshold=0.35) ===
              precision    recall  f1-score   support

    No Delay       0.97      0.77      0.86      8258
     Delayed       0.37      0.87      0.51      1238

    accuracy                           0.79      9496
   macro avg       0.67      0.82      0.69      9496
weighted avg       0.90      0.79      0.82      9496


=== Confusion Matrix (Threshold=0.35) ===
[[6395 1863]
 [ 167 1071]]


In [2]:
data[[
    'Departure Station', 'Arrival Destination', 'Month', 'Journey Day', 'Departure Hour', 'day number'
]]

,Departure Station,Arrival Destination,Month,Journey Day,Departure Hour,day number
0,London Paddington,Liverpool Lime Street,1,Normal Day,11,1
1,London Kings Cross,York,1,Normal Day,9,1
2,Liverpool Lime Street,Manchester Piccadilly,1,Normal Day,18,2
3,London Paddington,Reading,1,Normal Day,21,1
4,Liverpool Lime Street,London Euston,1,Normal Day,16,1
...,...,...,...,...,...,...
31648,Manchester Piccadilly,Liverpool Lime Street,4,Normal Day,20,30
31649,London Euston,Birmingham New Street,4,Normal Day,20,30
31650,Manchester Piccadilly,Liverpool Lime Street,4,Normal Day,20,30
31651,London Euston,Birmingham New Street,4,Normal Day,21,30


In [3]:
data.head()

,Transaction ID,Date of Purchase,Time of Purchase,Purchase Type,Payment Method,Railcard,Ticket Class,Ticket Type,Price,Departure Station,...,Old Price,refund,Discount,Journey Day,Month,km,Delay,Purchase Day,day number,Delayed?
0,da8a6ba8-b3dc-4677-b176,2023-12-08 12:41:11,12:41:11,Online,Contactless,Adult,Standard,Advance,43,London Paddington,...,129.0,0,86.0,Normal Day,1,283.732982,0.00,Friday,1,0
1,b0cdd1b0-f214-4197-be53,2023-12-16 11:23:01,11:23:01,Station,Credit Card,Adult,Standard,Advance,23,London Kings Cross,...,69.0,0,46.0,Normal Day,1,277.832076,0.08,Saturday,1,1
2,f3ba7a96-f713-40d9-9629,2023-12-19 19:51:27,19:51:27,Online,Credit Card,No Subscription,Standard,Advance,3,Liverpool Lime Street,...,6.0,0,3.0,Normal Day,1,50.182138,0.00,Tuesday,2,0
3,b2471f11-4fe7-4c87-8ab4,2023-12-20 23:00:36,23:00:36,Station,Credit Card,No Subscription,Standard,Advance,13,London Paddington,...,26.0,0,13.0,Normal Day,1,55.587780,0.00,Wednesday,1,0
4,2be00b45-0762-485e-a7a3,2023-12-27 18:22:56,18:22:56,Online,Contactless,No Subscription,Standard,Advance,76,Liverpool Lime Street,...,152.0,0,76.0,Normal Day,1,284.739823,0.00,Wednesday,1,0
